In [1]:
import os
import random
import string
import math

import hail as hl
from ukb_utils import hail_init
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from ukb_utils import genotypes
from ukb_utils import variants
from ko_utils import ko
from ko_utils import io
from ko_utils import samples
from ko_utils.ko import PLOF_CSQS, MISSENSE_CSQS, SYNONYMOUS_CSQS, OTHER_CSQS
import scipy.stats as stats

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Initializing Hail on SGE with 1 CPU(s).


2023-01-16 09:59:48 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2023-01-16 09:59:49 WARN  SparkConf:69 - Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Running on Apache Spark version 3.1.3
SparkUI available at http://compa014.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.102-817f6fb3468f
LOGGING: writing to logs/hail/hail_test_export.log


2023-01-16 10:06:30 Hail: INFO: Reading table to impute column types(0 + 1) / 1]
2023-01-16 10:06:36 Hail: INFO: Finished type imputation            (0 + 1) / 1]
  Loading field 'chr' as type int32 (imputed)
  Loading field 'pos' as type int32 (imputed)
  Loading field 'a0' as type str (imputed)
  Loading field 'a1' as type str (imputed)
  Loading field 'rsid' as type str (imputed)
  Loading field 'af_UKBB' as type float64 (imputed)
  Loading field 'ld' as type float64 (imputed)
  Loading field 'pos_hg17' as type str (imputed)
  Loading field 'pos_hg18' as type str (imputed)
  Loading field 'pos_hg38' as type str (imputed)
  Loading field 'group_id' as type int32 (imputed)


In [50]:
# load data and annotate
ht = hl.import_table("/well/lindgren/flassen/ressources/hapmap/ldpred2/map.txt.gz", force = True, impute = True)
ht = ht.annotate(grch37_varid = hl.delimit([hl.str(ht.chr), hl.str(ht.pos), hl.str(ht.a0), hl.str(ht.a1)], ':'))
ht = ht.key_by(**hl.parse_variant(ht.grch37_varid, reference_genome = 'GRCh37'))

# setup liftover references
from_build = "GRCh37"
to_build = "GRCh38"
liftover_path = variants.get_liftover_chain_path(from_build, to_build)
rg_from = hl.get_reference(from_build)
rg_to = hl.get_reference(to_build)
if not rg_from.has_liftover(rg_to):
    rg_from.add_liftover(liftover_path, rg_to)

if not rg_to.has_sequence():
    reference_path = get_reference_path("GRCh38")
    rg_to.add_sequence(*reference_path)

# do liftover and flip allele for negative strand
ht = ht.annotate(new_locus=hl.liftover(ht.locus, to_build, include_strand=True),old_locus=ht.locus)
ht = ht.filter(hl.is_defined(ht.new_locus))
ht = ht.annotate(new_alleles=hl.if_else(ht.new_locus.is_negative_strand, [hl.reverse_complement(ht.alleles[0]), hl.reverse_complement(ht.alleles[1])],ht.alleles))
ht = ht.key_by(locus=ht.new_locus.result, alleles=ht.new_alleles)

# check if the liftover was succesfull
ht = ht.annotate(
    ref_allele_mismatch = ht.new_locus.result.sequence_context() != ht.new_alleles[0],
    locus_fail_liftover=hl.is_missing(ht.new_locus.result)
)




2023-01-16 12:13:02 Hail: INFO: Reading table to impute column types
2023-01-16 12:13:07 Hail: INFO: Finished type imputation            (0 + 1) / 1]
  Loading field 'chr' as type int32 (imputed)
  Loading field 'pos' as type int32 (imputed)
  Loading field 'a0' as type str (imputed)
  Loading field 'a1' as type str (imputed)
  Loading field 'rsid' as type str (imputed)
  Loading field 'af_UKBB' as type float64 (imputed)
  Loading field 'ld' as type float64 (imputed)
  Loading field 'pos_hg17' as type str (imputed)
  Loading field 'pos_hg18' as type str (imputed)
  Loading field 'pos_hg38' as type str (imputed)
  Loading field 'group_id' as type int32 (imputed)


In [53]:
ht = ht.annotate(grch38_varid = hl.delimit([hl.str(ht.locus.contig), hl.str(ht.locus.position), hl.str(ht.alleles[0]), hl.str(ht.alleles[1])], ':'))
ht.describe()


----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'chr': int32 
    'pos': int32 
    'a0': str 
    'a1': str 
    'rsid': str 
    'af_UKBB': float64 
    'ld': float64 
    'pos_hg17': str 
    'pos_hg18': str 
    'pos_hg38': str 
    'group_id': int32 
    'grch37_varid': str 
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'new_locus': struct {
        result: locus<GRCh38>, 
        is_negative_strand: bool
    } 
    'old_locus': locus<GRCh37> 
    'new_alleles': array<str> 
    'ref_allele_mismatch': bool 
    'locus_fail_liftover': bool 
    'grch38_varid': str 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [48]:
#ht = ht.key_by("grch38_varid")
ht = ht.key_by(*[ht.locus, ht.alleles])
ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'grch37_varid': str 
    'grch38_varid': str 
    'rsid': str 
    'af_UKBB': float64 
    'ld': float64 
----------------------------------------
Key: ['locus', 'alleles']
----------------------------------------


In [43]:
ht = ht.select(*[ht.grch37_varid, ht.grch38_varid, ht.rsid, ht.af_UKBB, ht.ld])
ht.flatten().export(out_prefix + ".txt.gz")

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'grch37_varid': str 
    'grch38_varid': str 
    'rsid': str 
    'af_UKBB': float64 
    'ld': float64 
----------------------------------------
Key: []
----------------------------------------


In [24]:
def liftover_hailtable(ht, from_build = 'GRCh37', to_build = 'GRCh38'):
    r""" Liftover variants from one build to another
    :param mt: a HailTable
    :param from_build: Build to liftover from (either 'GRCh37' or 'GRCh38')
    :param to_build: Build to liftover to
    :param fix_ref
    :param drop_annotations: boolean indicating whether old locus and alleles should be dropped.
    """

  
    )
    


    return ht

In [11]:
ht = variants.liftover(ht, from_build='GRCh37', to_build='GRCh38', drop_annotations=True, fix_ref=True)

AttributeError: Table instance has no field, method, or property 'annotate_rows'
    Did you mean:
        Table methods: 'annotate', 'annotate_globals'

In [3]:
hm_path = "/well/lindgren/flassen/ressources/hapmap/weights.l2.ldscore.liftover.ht/"

In [10]:
ht = hl.read_table(hm_path)

In [12]:
p = "data/unphased/imputed/common_append_missing/ukb_imp_200k_common_append_missing_chr21.mt/"
mt = hl.read_matrix_table(p)
mt = mt.filter_rows(hl.is_defined(ht[mt.locus]))
mt = mt.count()

In [13]:
mt

(11624, 176587)

In [14]:
p = "data/unphased/imputed/common/ukb_imp_200k_common_chr21.mt"
mt = hl.read_matrix_table(p)
mt = mt.filter_rows(hl.is_defined(ht[mt.locus]))
mt.count()

(11624, 176266)

In [18]:
chrom = 21
extract = "data/phenotypes/phased_sample_list.txt"
extract2 = "/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/samples/09_final_qc.keep.sample_list"
min_maf=0.01
min_info=0.8
missing=0.10


In [16]:
# load chromosome
mfi = genotypes.get_ukb_imputed_v3_mfi(chrom)
mfi = mfi.annotate(chrom = chrom)

# annotate mfi
mfi = mfi.annotate(ref = hl.if_else(mfi.f6 == mfi.a1, mfi.a2, mfi.a1))
mfi = mfi.annotate(variant = hl.delimit([hl.str(mfi.chrom), hl.str(mfi.position), mfi.ref, mfi.f6], ':'))
mfi = mfi.key_by(**hl.parse_variant(mfi.variant,  reference_genome= 'GRCh37'))

# load imputed
mt = genotypes.get_ukb_imputed_v3_bgen([chrom])
#mt = mt.repartition(64)
mt = mt.annotate_rows(info_score = mfi[mt.row_key].info)
mt = mt.select_entries(mt.GT)




2023-01-14 13:48:32 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)
2023-01-14 13:48:33 Hail: INFO: Number of BGEN files parsed: 1
2023-01-14 13:48:33 Hail: INFO: Number of samples in BGEN files: 487409
2023-01-14 13:48:33 Hail: INFO: Number of variants across all BGEN files: 1261158


In [19]:
# Filter to relevant samples
if extract:
    ht_final_samples = hl.import_table(
        extract, no_header=True, key='f0', delimiter=',')
    mt = mt.filter_cols(hl.is_defined(ht_final_samples[mt.col_key]))

# Filter to relevant samples
if extract2:
    ht_final_samples2 = hl.import_table(
        extract2, no_header=True, key='f0', delimiter=',')
    mt = mt.filter_cols(hl.is_defined(ht_final_samples2[mt.col_key]))


2023-01-14 13:48:56 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
2023-01-14 13:48:57 Hail: INFO: Loading <StructExpression of type struct{f0: str, f1: str, f2: str, f3: str, f4: str, f5: str, f6: str, f7: str, f8: str, f9: str, f10: str, f11: str, f12: str, f13: str, f14: str, f15: str, f16: str, f17: str, f18: str, f19: str, f20: str, f21: str, f22: str, f23: str, f24: str, f25: str, f26: str, f27: str, f28: str, f29: str, f30: str, f31: str, f32: str, f33: str, f34: str, f35: str, f36: str, f37: str, f38: str, f39: str, f40: str, f41: str, f42: str, f43: str, f44: str, f45: str, f46: str, f47: str, f48: str, f49: str, f50: str, f51: str, f52: str, f53: str, f54: str, f55: str, f56: str, f57: str, f58: str, f59: str, f60: str, f61: str, f62: str, f63: str, f64: str, f65: str, f66: str, f67: str, f68: str, f69: str, f70: str, f71: str, f72: str, f73: str, f74: str, f75: str, f76: str, f77: str, f78: str, f79: str, f80: str, f81: str

In [54]:
from ukb_utils import hail_init
from ukb_utils import variants

#os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
#hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

path="/well/lindgren/flassen/ressources/hapmap/weights.l2.ldscore.tsv.gz"
ht = hl.import_table(path, force = True, impute = True)

ht = ht.annotate(variant = hl.delimit([hl.str(ht.CHR), hl.str(ht.BP), 'A', 'T'], ':'))
#ht = ht.key_by(**hl.parse_variant(ht.variant, reference_genome = 'GRCh37'))

2023-01-15 19:18:09 Hail: INFO: Reading table to impute column types
2023-01-15 19:18:13 Hail: INFO: Finished type imputation            (0 + 1) / 1]
  Loading field 'CHR' as type int32 (imputed)
  Loading field 'SNP' as type str (imputed)
  Loading field 'BP' as type int32 (imputed)
  Loading field 'CM' as type float64 (imputed)
  Loading field 'MAF' as type float64 (imputed)
  Loading field 'L2' as type float64 (imputed)


In [55]:
ht.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'CHR': int32 
    'SNP': str 
    'BP': int32 
    'CM': float64 
    'MAF': float64 
    'L2': float64 
----------------------------------------
Key: []
----------------------------------------


In [48]:
hm_path = "/well/lindgren/flassen/ressources/hapmap/weights.l2.ldscore.liftover.ht/"
hm = hl.read_table(hm_path)
hm = hm.key_by(hm.locus_grch37)

In [49]:
mt = hl.read_matrix_table("test/newest_checkpoint.mt")

In [50]:
mt = mt.filter_rows(hl.is_defined(hm[mt.locus].locus)) # GRCh37

In [51]:
maf_expr = variants.get_maf_expr(mt) >= 0.01
mt = mt.filter_rows(maf_expr)
mt.count()

2023-01-15 18:39:19 Hail: INFO: Ordering unsorted dataset with network shuffle1]


(16714, 176266)

In [ ]:
mt = mt.filter_rows(mt.info_score >= 0.5)
mt.count()

2023-01-15 18:46:07 Hail: INFO: Ordering unsorted dataset with network shuffle1]


In [46]:
missing_expr = variants.get_missing_expr(mt) <= 0.01
mt = mt.filter_rows(missing_expr)
mt.count()

2023-01-15 18:32:24 Hail: INFO: Ordering unsorted dataset with network shuffle1]


(11624, 176266)

In [47]:
#mt = hl.read_matrix_table("test/newest_checkpoint.mt")
#maf_expr = variants.get_maf_expr(mt) >= 0.01
#info_expr = mt.info_score >= 0.8
#missing_expr = variants.get_missing_expr(mt) <= 0.01
#mt = mt.filter_rows((maf_expr) & (info_expr) & (missing_expr))
#mt.count()

In [ ]:
#maf_expr = variants.get_maf_expr(mtt) >= 0.005
#mtt = mtt.filter_rows(maf_expr)
#mtt = mtt.repartition(16)
#mtt = mtt.checkpoint("test/newest_checkpoint.mt")

In [25]:

mt.count()

KeyboardInterrupt: 

In [ ]:
# perform variant filtering after subsetting samples
info_expr = mt.info_score >= min_info

mt = mt.filter_rows(info_expr & maf_expr)
missing_expr = variants.get_missing_expr(mt) <= missing
mt = mt.filter_rows(missing_expr)
mt.count()

In [63]:
# load clinvar
clinvar_path = "/well/lindgren/flassen/ressources/clinvar/ftp//clinvar_20230107.vcf.gz"
recoding = {str(i):"chr" + str(i) for i in range(1,23)}
mt1 = hl.import_vcf(clinvar_path, contig_recoding = recoding, force = True, skip_invalid_loci  = True)
ht1 = mt1.rows()

In [65]:
# load data 
input_path = "data/mt/prefilter/pp90/ukb_wes_union_calls_200k_chr21.loftee.worst_csq_by_gene_canonical.pp90.maf0_005.mt"
input_type = "mt"
mt2 = io.import_table(input_path, input_type, calc_info = False)

In [72]:
ht2 = mt2.rows()
ht2 = ht2.select(*[ht2.info, ht2.consequence.vep.worst_csq_by_gene_canonical])
ht2 = ht2.annotate(clinvar = ht1[ht2.key].info)

In [74]:

ht2.flatten().describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'info.AF': float64 
    'info.AC': int32 
    'info.AN': int32 
    'worst_csq_by_gene_canonical.allele_num': int32 
    'worst_csq_by_gene_canonical.amino_acids': str 
    'worst_csq_by_gene_canonical.biotype': str 
    'worst_csq_by_gene_canonical.canonical': int32 
    'worst_csq_by_gene_canonical.ccds': str 
    'worst_csq_by_gene_canonical.cdna_start': int32 
    'worst_csq_by_gene_canonical.cdna_end': int32 
    'worst_csq_by_gene_canonical.cds_end': int32 
    'worst_csq_by_gene_canonical.cds_start': int32 
    'worst_csq_by_gene_canonical.codons': str 
    'worst_csq_by_gene_canonical.consequence_terms': array<str> 
    'worst_csq_by_gene_canonical.distance': int32 
    'worst_csq_by_gene_canonical.domains': array<struct {
        db: str, 
        name: str
    }> 
    'worst_csq_by_gene_canonical.exon'

In [4]:
intervals="data/conditional/common/intervals/min_mac4/ukb_eur_wes_200k_PSOR_combined_pLoF_damaging_missense.ht"

In [5]:
# get chromosomes and ranges to extract
ht = hl.read_table(intervals)
chromosomes = [x.replace('chr', '') for x in list(set(ht.contig.collect()))]
hail_intervals = ht.intervals.collect()



In [12]:
df = ht.to_pandas()
df

,gene,pvalue,contig,start,end,contig_length,end_with_padding,start_with_padding,start_valid,end_valid,valid_intervals,intervals
0,ENSG00000113088,6.936591e-07,chr5,55024256,55034570,181538259,56034570,54024256,True,True,True,[chr5:54024256-56034570]
1,ENSG00000114302,1.004644e-07,chr3,48744597,48847874,198295559,49847874,47744597,True,True,True,[chr3:47744597-49847874]
2,ENSG00000136110,1.578844e-06,chr13,52703264,52739820,114364328,53739820,51703264,True,True,True,[chr13:51703264-53739820]
3,ENSG00000154252,1.672486e-07,chr2,241776822,241804287,242193529,242193529,240776822,True,True,True,[chr2:240776822-242193529]
4,ENSG00000164318,2.72265e-06,chr5,38258409,38465480,181538259,39465480,37258409,True,True,True,[chr5:37258409-39465480]
5,ENSG00000173480,3.403649e-06,chr19,57900296,57916610,58617616,58617616,56900296,True,True,True,[chr19:56900296-58617616]
6,ENSG00000173612,3.738442e-09,chr6,116792085,116829083,170805979,117829083,115792085,True,True,True,[chr6:115792085-117829083]
7,ENSG00000204520,3.28096e-10,chr6,31399784,31415315,170805979,32415315,30399784,True,True,True,[chr6:30399784-32415315]


In [31]:
for idx, row in df.iterrows():
    # iterate over rows
    gene = row['gene']
    contig = row['contig']
    start = row['start_with_padding']
    end = row['end_with_padding']
    interval = f"{contig}:{start}-{end}"
    hail_interval = hl.parse_locus_interval(interval, reference_genome='GRCh38')
    # create interval per region
    path = f"data/unphased/imputed/common_append_missing/ukb_imp_200k_common_append_missing_{contig}.mt"
    mt = hl.read_matrix_table(path)
    mt = hl.filter_intervals(mt, hl.literal([hail_interval]))
    
    

In [ ]:
# get all the relevant chromsomes
mts = list()
for chrom in chromosomes:
    
    mt = hl.read_matrix_table(path)
    mts.append(mt)

# combine them by variants (across chroms)
mt = mts[0].union_rows(*mts[1:])
mt = hl.filter_intervals(mt, hail_intervals)

In [3]:
path1 = "data/conditional/rare/combined/mt/old/ukb_eur_wes_200k_chr21_pLoF_damaging_missense.mt/"
mt1 = hl.read_matrix_table(path1)

FatalError: HailException: No file or directory found at data/conditional/rare/combined/mt/old/ukb_eur_wes_200k_chr21_pLoF_damaging_missense.mt/

Java stack trace:
is.hail.utils.HailException: No file or directory found at data/conditional/rare/combined/mt/old/ukb_eur_wes_200k_chr21_pLoF_damaging_missense.mt/
	at is.hail.utils.ErrorHandling.fatal(ErrorHandling.scala:17)
	at is.hail.utils.ErrorHandling.fatal$(ErrorHandling.scala:17)
	at is.hail.utils.package$.fatal(package.scala:78)
	at is.hail.expr.ir.RelationalSpec$.readMetadata(AbstractMatrixTableSpec.scala:33)
	at is.hail.expr.ir.RelationalSpec$.readReferences(AbstractMatrixTableSpec.scala:74)
	at is.hail.variant.ReferenceGenome$.fromHailDataset(ReferenceGenome.scala:581)
	at is.hail.variant.ReferenceGenome.fromHailDataset(ReferenceGenome.scala)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl

In [4]:
rsid1 = mt1.rsid.collect()

In [5]:
path2 = "data/mt/prefilter/pp90/ukb_wes_union_calls_200k_chr21.loftee.worst_csq_by_gene_canonical.pp90.maf0_005.mt/"
mt2 = hl.read_matrix_table(path2)

In [6]:
rsid2 = mt2.varid.collect()

In [7]:
len(rsid2)

141807

In [8]:
mt2 = mt2.filter_rows(hl.literal(rsid1).contains(mt2.varid))

In [9]:
mt2.count()

(5603, 176587)

In [10]:
lof = mt2.consequence.vep.worst_csq_by_gene_canonical.lof.collect()

In [17]:
print(lof.count("LC"))
print(lof.count("HC"))

233
1559


In [5]:
calls_path = "data/unphased/calls/liftover/ukb_liftover_calls_500k_chr21.mt/"
calls = hl.read_matrix_table(calls_path)

In [16]:
calls.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'ref_allele_mismatch': bool
    'locus_fail_liftover': bool
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [13]:
imp.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'varid': str
    'info_score': float64
    'new_locus': struct {
        result: locus<GRCh38>, 
        is_negative_strand: bool
    }
    'old_locus': locus<GRCh37>
    'new_alleles': array<str>
    'ref_allele_mismatch': bool
    'locus_fail_liftover': bool
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [15]:
calls.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'ref_allele_mismatch': bool
    'locus_fail_liftover': bool
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [29]:
hl.missing(hl.call())

TypeError: missing: parameter 't': expected (hail.expr.types.HailType or (str) or NoneType), found hail.expr.expressions.typed_expressions.CallExpression: <CallExpression of type call>

In [36]:
hl.call().show()

""
""
call
-


In [ ]:
ht_ko = hl.import_table("data/phenotypes/samples/sample_list_phased.txt", no_header = True, key = 'f0')
ht_qc = hl.import_table("data/phenotypes/samples/sample_list_final.qc.txt", no_header = True, key = 'f0')

In [19]:
calls = calls.filter_cols(hl.is_defined(ht_phased[calls.col_key]))
calls = calls.filter_cols(hl.is_defined(ht_qc[calls.col_key]))
calls = calls.filter_cols(~hl.is_defined(imp.cols()[calls.col_key]))

In [20]:
calls.count_cols()

FatalError: OutOfMemoryError: Java heap space

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 11.0 failed 1 times, most recent failure: Lost task 0.0 in stage 11.0 (TID 11) (compa041.hpc.in.bmrc.ox.ac.uk executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.util.Arrays.copyOf(Arrays.java:3236)
	at java.io.ByteArrayOutputStream.grow(ByteArrayOutputStream.java:118)
	at java.io.ByteArrayOutputStream.ensureCapacity(ByteArrayOutputStream.java:93)
	at java.io.ByteArrayOutputStream.write(ByteArrayOutputStream.java:153)
	at org.apache.spark.util.ByteBufferOutputStream.write(ByteBufferOutputStream.scala:41)
	at java.io.ObjectOutputStream$BlockDataOutputStream.write(ObjectOutputStream.java:1853)
	at java.io.ObjectOutputStream.write(ObjectOutputStream.java:709)
	at org.apache.spark.util.Utils$.writeByteBuffer(Utils.scala:233)
	at org.apache.spark.scheduler.DirectTaskResult.$anonfun$writeExternal$1(TaskResult.scala:53)
	at org.

In [49]:
input_path = "data/unphased/imputed/common_8cores/ukb_imp_200k_common_chr20.mt"

In [50]:
mt = hl.read_matrix_table(input_path)
mt.count_cols()

198895

In [35]:
# what are the sample overlaps
imputed = genotypes.get_ukb_imputed_v3_bgen([21])
calls = genotypes.get_ukb_genotypes_bed([21])
cols_imputed = imputed.s.collect()
cols_calls = calls.s.collect()
print(len(cols_imputed))
print(len(cols_calls))

ht = hl.import_table("data/knockouts/alt/samples.txt", no_header = True)
cols_ko = ht.f0.collect()

print(len(cols_ko))
print(len(list(set(cols_imputed) & set(cols_ko))))
print(len(list(set(cols_calls) & set(cols_ko))))
print(len(list(set(cols_calls) & set(cols_ko) & set(cols_ko))))

2022-12-21 08:49:39 Hail: INFO: Number of BGEN files parsed: 1
2022-12-21 08:49:39 Hail: INFO: Number of samples in BGEN files: 487409
2022-12-21 08:49:39 Hail: INFO: Number of variants across all BGEN files: 1261158
2022-12-21 08:50:06 Hail: INFO: Found 488377 samples in fam file.
2022-12-21 08:50:06 Hail: INFO: Found 11342 variants in bim file.


487409
488377


2022-12-21 08:50:55 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)


176587
176266
176587
176587


In [36]:
imputed = genotypes.get_ukb_imputed_v3_bgen([21])
calls = genotypes.get_ukb_genotypes_bed([21])

2022-12-21 09:03:19 Hail: INFO: Number of BGEN files parsed: 1
2022-12-21 09:03:19 Hail: INFO: Number of samples in BGEN files: 487409
2022-12-21 09:03:19 Hail: INFO: Number of variants across all BGEN files: 1261158
2022-12-21 09:03:47 Hail: INFO: Found 488377 samples in fam file.
2022-12-21 09:03:47 Hail: INFO: Found 11342 variants in bim file.


In [37]:
ht = imputed.s

In [40]:
ht.export("test.txt", header = False)

2022-12-21 09:06:35 Hail: INFO: merging 16 files totalling 3.7M...(15 + 1) / 16]
2022-12-21 09:06:35 Hail: INFO: while writing:
    test.txt
  merge time: 32.686ms


176587

In [18]:
def count_gt_entries(mt: hl.MatrixTable):
    """sum phased/unphased calls 
    :param ht: MatrixTable with gts entry (see collect_gt_by_expr)
    """
    assert 'gts' in list(mt.entry), 'missing entry field "gts"'
   
    return (mt.annotate_entries( 
            hom_alt_n = mt.GT.is_hom_var(),
            phased=hl.struct(
                a1=(mt.GT == hl.parse_call('1|0')),
                a2=(mt.GT == hl.parse_call('0|1'))
            ),
            unphased=hl.struct(
                n=((mt.GT.phased) & (mt.GT.is_het_ref()))
            )
        )
    )

In [16]:
ko.sum_gts_entries(ko.collect_phase_count_by_expr(mt, gene_expr)).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'gts': array<call>
    'varid': array<str>
    'hom_alt_n': int32
    'phased': struct {
        a1: int32, 
        a2: int32
    }
    'unphased': struct {
        n: int32
    }
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [14]:
ko.aggr_phase_count_by_expr(mt, gene_expr).describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'phased': struct {
        a1: int64, 
        a2: int64, 
        n: int64
    }
    'unphased': struct {
        n: int64
    }
    'hom_alt_n': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [19]:
count_gt_entries(mt, gene_expr).describe()

TypeError: count_gt_entries() takes 1 positional argument but 2 were given

In [7]:
gene_expr = mt.consequence.vep.worst_csq_by_gene_canonical.gene_id
genes = ko.aggr_phase_count_by_expr(mt, gene_expr)
genes = genes.repartition(64)
#expr_pko = ko.calc_prob_ko(genes.hom_alt_n, genes.phased, genes.unphased, only_homs=False)
#expr_ko = ko.annotate_knockout(genes.hom_alt_n, expr_pko)

In [8]:
genes = genes.checkpoint("hail_checkpoint_delete.mt", overwrite = True)

2022-12-07 11:59:16 Hail: INFO: Ordering unsorted dataset with network shuffle2]
2022-12-07 11:59:46 Hail: INFO: wrote matrix table with 142 rows and 176587 columns in 64 partitions to hail_checkpoint_delete.mt
    Total size: 1.85 MiB
    * Rows/entries: 716.36 KiB
    * Columns: 1.15 MiB
    * Globals: 11.00 B
    * Smallest partition: 2 rows (7.86 KiB)
    * Largest partition:  2 rows (22.74 KiB)


In [22]:
genes.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'phased': struct {
        a1: int64, 
        a2: int64, 
        n: int64
    }
    'unphased': struct {
        n: int64
    }
    'hom_alt_n': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [27]:
genes.filter_entries(genes.phased.a1 > 0).show()

+-------------------+---------------------+---------------------+
| gene_id           | '1705888'.phased.a1 | '1705888'.phased.a2 |
+-------------------+---------------------+---------------------+
| str               |               int64 |               int64 |
+-------------------+---------------------+---------------------+
| "ENSG00000141956" |                  NA |                  NA |
| "ENSG00000141959" |                  NA |                  NA |
| "ENSG00000142149" |                  NA |                  NA |
| "ENSG00000142156" |                  NA |                  NA |
| "ENSG00000142166" |                  NA |                  NA |
| "ENSG00000142168" |                  NA |                  NA |
| "ENSG00000142173" |                  NA |                  NA |
| "ENSG00000142182" |                  NA |                  NA |
| "ENSG00000142185" |                  NA |                  NA |
| "ENSG00000142188" |                  NA |                  NA |
+-------------------+---------------------+---------------------+

+--------------------+----------------------+---------------------+
| '1705888'.phased.n | '1705888'.unphased.n | '1705888'.hom_alt_n |
+--------------------+----------------------+---------------------+
|              int64 |                int64 |               int64 |
+--------------------+----------------------+---------------------+
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
|                 NA |                   NA |                  NA |
+--------------------+----------------------+---------------------+
showing top 10 rows
showing the first 1 of 176587 columns

In [4]:
ht = hl.import_table(phenotypes,
             types={'eid': hl.tstr},
             missing=["",'""',"NA"],
             impute=True,
             force=True,
             key='eid'
             )

2022-11-14 10:47:54 Hail: INFO: Reading table to impute column types(0 + 1) / 1]
2022-11-14 10:48:37 Hail: INFO: Loading <StructExpression of type struct{eid: str, sex: int32, age: int32, `ukbb.centre`: int32, PC1: float64, PC2: float64, PC3: float64, PC4: float64, PC5: float64, PC6: float64, PC7: float64, PC8: float64, PC9: float64, PC10: float64, PC11: float64, PC12: float64, PC13: float64, PC14: float64, PC15: float64, PC16: float64, PC17: float64, PC18: float64, PC19: float64, PC20: float64, PC21: float64, PC22: float64, PC23: float64, PC24: float64, PC25: float64, PC26: float64, PC27: float64, PC28: float64, PC29: float64, PC30: float64, PC31: float64, PC32: float64, PC33: float64, PC34: float64, PC35: float64, PC36: float64, PC37: float64, PC38: float64, PC39: float64, PC40: float64, BC_combined: bool, BC_combined_primary_care: bool, CAD_combined: bool, CAD_combined_primary_care: bool, COPD_combined: bool, COPD_combined_primary_care: bool, CLD_combined: bool, CLD_combined_primary

NameError: name 'mt' is not defined

In [10]:
mt = io.import_table(input_path, input_type)
mt = mt.annotate_cols(pheno=ht[mt.s])

2022-11-14 10:49:32 Hail: INFO: Found 246271 samples in fam file.
2022-11-14 10:49:32 Hail: INFO: Found 16727 variants in bim file.


In [15]:
"CIRR_combined_primary_care" in list(mt.pheno)

True

In [17]:
defined = hl.is_defined(mt.pheno.CIRR_combined_primary_care).collect()

In [11]:
from ukb_utils import variants

In [3]:
p1 = "data/phased/wes_union_calls/200k/shapeit5/phase_rare/ukb_wes_union_calls_shapeit5_200k_chr21-20xshort/shapeit5_prs100000_pro25000_mprs150000.1of1.vcf.gz"
p2 = "data/prephased/wes_union_calls/ukb_wes_union_calls_200k_chr21.vcf.gz"

In [17]:
mt1 = io.import_table(p1, "vcf")
mt2 = io.import_table(p2, "vcf")

In [18]:
mt1 = mt1.annotate_entries(
    GT_rb = mt2[mt1.row_key, mt1.col_key].GT,
    PS_rb = mt2[mt1.row_key, mt1.col_key].PS
)

In [22]:
mt1.transmute_rows(
    info = mt1.info.annotate(
        AC_rb = mt2.rows()[mt1.row_key].info.AC,
        AN_rb = mt2.rows()[mt1.row_key].info.AN
    )
)

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AC: int32, 
        AN: int32, 
        AC_rb: int32, 
        AN_rb: int32
    }
----------------------------------------
Entry fields:
    'GT': call
    'PP': float64
    'GT_rb': call
    'PS_rb': int32
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [16]:
mt1.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AC: int32, 
        AN: int32
    }
    'AC_rb': int32
    'AN_rb': int32
----------------------------------------
Entry fields:
    'GT': call
    'PP': float64
    'GT_rb': call
    'PS_rb': int32
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [20]:
mt1 = mt1.filter_rows(variants.get_mac_expr(mt1) > 1 )
mt2 = mt2.filter_rows(variants.get_mac_expr(mt2) > 1 )

In [21]:
print(mt1.count())
print(mt2.count())

(10700, 176618)


(10707, 185492)


In [22]:
mt1 = mt1.annotate_rows(mismatch = variants.get_reference_mismatch_expr(mt1.locus, mt1.alleles))
mt2 = mt2.annotate_rows(mismatch = variants.get_reference_mismatch_expr(mt2.locus, mt2.alleles))

In [23]:
print(mt1.aggregate_rows(hl.agg.sum(mt1.mismatch)))

1


In [24]:
print(mt2.aggregate_rows(hl.agg.sum(mt2.mismatch)))

0


In [ ]:
input_path="data/prs/hapmap/ukb_500k/fitting/ukb_hapmap_500k_eur_chrCHR"
input_type="plink"
phenotypes="/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/curated_phenotypes_cts.tsv"
phenotypes="Albumin_residual"
covariates=''
100
data/prs/sumstat/cts/ukb_hapmap_500k_eur_Albumin_residual_chrCHR


In [7]:
input_path = "data/mt/annotated/ukb_eur_wes_union_calls_200k_chr19.mt"

In [8]:
mt = io.import_table(input_path, "mt", calc_info = False)

In [22]:
mt.consequence.vep.worst_csq_for_variant_canonical.most_severe_consequence

<StringExpression of type str>

In [18]:
mt.consequence.vep.worst_csq_by_gene_canonical

<ArrayStructExpression of type array<struct{allele_num: int32, amino_acids: str, biotype: str, canonical: int32, ccds: str, cdna_start: int32, cdna_end: int32, cds_end: int32, cds_start: int32, codons: str, consequence_terms: array<str>, distance: int32, domains: array<struct{db: str, name: str}>, exon: str, gene_id: str, gene_pheno: int32, gene_symbol: str, gene_symbol_source: str, hgnc_id: str, hgvsc: str, hgvsp: str, hgvs_offset: int32, impact: str, intron: str, lof: str, lof_flags: str, lof_filter: str, lof_info: str, minimised: int32, polyphen_prediction: str, polyphen_score: float64, protein_end: int32, protein_start: int32, protein_id: str, sift_prediction: str, strand: int32, swissprot: str, transcript_id: str, trembl: str, uniparc: str, variant_allele: str, revel_score: float64, cadd_phred: float64, most_severe_consequence: str, csq_score: float64}>>

In [3]:
mt = io.import_table("data/unphased/wes_union_calls/bcftools/newtest/tagged.vcf.bgz", "vcf")

In [5]:
mt = mt.annotate_rows(
    missing = mt.aggregate_entries(hl.agg.sum(hl.is_defined(mt.GT)))
)

2022-11-04 15:54:49 Hail: INFO: scanning VCF for sortedness...
2022-11-04 16:00:11 Hail: INFO: Coerced sorted VCF - no additional import work to do


In [6]:
x = mt.missing.collect()

In [10]:
q = [27325787351-y for y in x]